# Realtor Dataset – Exploratory Data Analysis

This notebook documents the EDA performed on the Realtor real-estate dataset.

**Goal:** understand data quality, distributions, missing values, outliers and relationships between variables before building a clustering model.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

df = pd.read_csv("data/realtor-data.csv")

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
df.head()

## 1. Dataset overview

The dataset contains information about properties, including price, number of bedrooms and bathrooms, lot size, house size and geographic information.

In [ ]:
df.info()
print("\nData types:")
print(df.dtypes)

## 2. Missing values

A large part of the dataset contains missing values. Instead of immediately removing rows, we first measure the missingness and later decide how each feature should be handled.

In [ ]:
missing = (
    df.isna()
      .sum()
      .sort_values(ascending=False)
)

missing_pct = (missing / len(df) * 100).round(2)

missing_table = pd.DataFrame({
    "missing": missing,
    "missing_pct": missing_pct,
})

missing_table

## 3. Duplicates

We check whether the dataset contains completely duplicated rows.

In [ ]:
duplicates = df.duplicated().sum()
print(f"Duplicate rows: {duplicates:,}")

## 4. Categorical distributions

`status` contains the property's current status. We inspect its distribution because a heavily imbalanced categorical variable may not be suitable as a primary clustering feature.

In [ ]:
print(df["status"].value_counts(dropna=False))
print()
print((df["status"].value_counts(normalize=True, dropna=False) * 100).round(2))

## 5. Cardinality

High-cardinality variables such as street, city and ZIP code need special consideration. Using all of them directly in a first K-Means model would create a very large feature space and could make the clusters difficult to interpret.

In [ ]:
df.nunique(dropna=False).sort_values(ascending=False)

## 6. Numeric distributions

We inspect descriptive statistics and quantiles to identify skewness and extreme values.

In [ ]:
numeric_columns = [
    "price",
    "bed",
    "bath",
    "acre_lot",
    "house_size",
]

df[numeric_columns].describe(percentiles=[0.5, 0.9, 0.95, 0.99, 0.995, 0.999]).T

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(10, 18))

for ax, column in zip(axes, numeric_columns):
    ax.hist(df[column].dropna(), bins=100)
    ax.set_title(column)
    ax.set_xlabel(column)
    ax.set_ylabel("Count")

plt.tight_layout()
plt.show()

## 7. Outlier analysis

The raw dataset contains clearly suspicious extreme values. Examples include extremely high numbers of bedrooms or bathrooms, very large house sizes, very large lot sizes and non-positive prices.

These values are investigated before clustering because K-Means is distance-based and can be strongly affected by extreme observations.

In [ ]:
outlier_checks = {
    "bed > 10": (df["bed"] > 10).sum(),
    "bath > 10": (df["bath"] > 10).sum(),
    "acre_lot > 100": (df["acre_lot"] > 100).sum(),
    "house_size > 10000": (df["house_size"] > 10000).sum(),
    "price > 10000000": (df["price"] > 10000000).sum(),
    "price <= 0": (df["price"] <= 0).sum(),
}

pd.Series(outlier_checks, name="count")

### Chosen cleaning thresholds

For the clustering preparation we remove:

- `price <= 0`
- `bed > 20`
- `bath > 20`
- `house_size > 20,000`
- `acre_lot > 1,000`

We do not automatically remove very expensive properties solely because their price is high; a high price can represent a real property rather than a data error.

## 8. Correlation analysis

We inspect correlations between the main numeric property variables. This helps us understand relationships and potential redundancy before selecting clustering features.

In [ ]:
corr = df[numeric_columns].corr()

corr

In [ ]:
plt.figure(figsize=(8, 6))
plt.imshow(corr, interpolation="nearest")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=45, ha="right")
plt.yticks(range(len(corr.columns)), corr.columns)
plt.colorbar(label="Correlation")
plt.title("Correlation matrix")
plt.tight_layout()
plt.show()

## 9. Feature selection for clustering

The first clustering model uses:

- `price`
- `bed`
- `bath`
- `acre_lot`
- `house_size`

These variables describe the property itself and its market value.

`street` and `brokered_by` are treated as identifier-like variables and are excluded.

`city` and `zip_code` have high cardinality and are excluded from the first numerical K-Means model. They can instead be used later to analyse the geographic composition of the resulting clusters.

`prev_sold_date` has substantial missingness, so it is not included in the primary clustering feature set. `status` is categorical and strongly imbalanced, so it is also better suited for post-clustering analysis than as a primary K-Means feature.

## 10. EDA conclusions

The EDA shows that:

1. The dataset is very large, with more than 2.2 million observations.
2. Several variables contain substantial missingness.
3. `price`, `acre_lot` and `house_size` are strongly right-skewed.
4. The raw data contains extreme and likely invalid observations.
5. `bed`, `bath` and `house_size` have clear positive relationships.
6. High-cardinality geographic and identifier-like variables should not be included directly in the first K-Means model.
7. The selected numerical features require missing-value handling, log transformation for strongly skewed variables and standardisation before K-Means.

The next step is therefore clustering preparation and selection of an appropriate number of clusters.